# The Use of PINN in Modeling of Thermoelectric Modules

**Paper:** Kluger, R., Buchalik, R., Nowak, I. (2026). *The Use of PINN in Modeling of Thermoelectric Modules.* Energies, 19(4), 878.

**Carpeta origen:** `PINNs/2. Termidinamica y cinetica/The_Use_of_PINN_in_Modeling_of_Thermoelectric_Modu.pdf`

## Como se usan las PINNs en este paper

El paper modela un sistema termoelectrico **de dos etapas** (Fig. 2): dos modulos Peltier en cascada entre un deposito termico superior, uno interno y uno inferior, sujetos a corrientes electricas dependientes del tiempo $I_1(t), I_2(t)$. Las ecuaciones (1)-(4) describen los flujos de calor por los efectos Seebeck, Peltier y Joule:

$$\dot Q_u=k_1(T_u-T_{u_{ir}})+T_u\alpha_1 I_1(t)-\tfrac12 R_1I_1(t)^2,\qquad \dot Q_{u_{ir}}=k_1(T_u-T_{u_{ir}})+T_{u_{ir}}\alpha_1I_1(t)+\tfrac12R_1I_1(t)^2$$
$$\dot Q_{l_{ir}}=k_2(T_{l_{ir}}-T_l)+T_{l_{ir}}\alpha_2I_2(t)-\tfrac12R_2I_2(t)^2,\qquad \dot Q_l=k_2(T_{l_{ir}}-T_l)+T_l\alpha_2I_2(t)+\tfrac12R_2I_2(t)^2$$

con balances de energia en los depositos capacitivos (Eq. 7-9) y relaciones de resistencia termica en cada contacto (Eq. 10-13). La PINN es una red $t\mapsto(T_u,T_{u_{ir}},T_{c_{ir}},T_{l_{ir}},T_l,T_{c_u},T_{c_l})$ (Tabla 1: 7 salidas, una por cada temperatura de estado), entrenada minimizando el funcional fisico (la formula clave del paper):

$$\Phi_{phys}=\sum_{i=1}^{7}w_i\sum_{t_j\in\Omega_{coll}}\big|\text{Res}_i\big(T(t_j),\dot T(t_j)\big)\big|^2$$

donde los 7 residuos corresponden a las Eq. (7)-(13) (3 EDOs de balance de energia + 4 relaciones algebraicas de resistencia termica), mas la condicion inicial $T_k(0)=T_k^{(0)}$ (Eq. 14) anadida como termino de penalizacion adicional. El objetivo del paper es modelar el efecto de **sobre-enfriamiento (supercooling)**: con una corriente periodica (pulsada) bien elegida, el sistema alcanza temperaturas transitoriamente mas bajas que el minimo posible en estado estacionario, explotando la inercia termica de los depositos capacitivos.

Este cuaderno reproduce fielmente la arquitectura de 7 salidas, las 13 ecuaciones del modelo, el funcional fisico $\Phi_{phys}$ con los 7 residuos y la condicion inicial, usando una forma de onda de corriente periodica pulsada representativa (el paper trata $I_1(t),I_2(t)$ como datos de entrada ya optimizados en trabajos previos, no reproducidos aqui).

**Nota:** el paper no reporta explicitamente los valores numericos de $k_1,\alpha_1,R_1,c_u,r_{uu}$, etc. en las paginas revisadas (estarian en una tabla posterior de parametros de simulacion); aqui se usan valores representativos tipicos de modulos Peltier comerciales, documentados explicitamente en el codigo.

## Repositorio publico de referencia

El paper menciona explicitamente la biblioteca **deepxde** (Lu et al.) como la herramienta usada para implementar las PINNs: "a Python library was proposed in [4] that allows the practical use of PINNs. That library called `deepxde` was used in this work."

- **lululxvi/deepxde** &mdash; https://github.com/lululxvi/deepxde

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Parametros (representativos de modulos Peltier), forma de onda de corriente pulsada y condiciones iniciales (Eq. 14)

In [ ]:
k1 = k2 = 0.6      # conductancia termica del modulo [W/K]
alpha1 = alpha2 = 0.05   # coeficiente de Seebeck [V/K]
R1 = R2 = 2.0       # resistencia electrica del modulo [Ohm]
c_u, c_ir, c_l = 4.0, 3.0, 4.0        # capacidades termicas de los depositos [J/K]
r_uu = r_ul = r_lu = r_ll = 0.3       # resistencias termicas de contacto [K/W]
W_u, W_l = 0.05, 2.0    # coeficientes de intercambio con el ambiente [W/K] (W_l >> W_u, Seccion 3)
T0 = 25.0            # temperatura ambiente [C]
t_max = 60.0          # horizonte de simulacion [s]

I_amp, I_period, I_duty = 1.5, 6.0, 0.4  # amplitud, periodo y ciclo de trabajo del pulso de corriente
sharpness = 30.0
norm_factor = torch.sigmoid(torch.tensor(sharpness * I_duty / 2)) ** 2

def current_waveform(t):
    """Forma de onda periodica pulsada representativa (el paper trata I(t) como dato de entrada
    ya optimizado en trabajos previos [13,14], no reproducidos en detalle aqui). Se usa una version
    suavizada (producto de sigmoides) del pulso rectangular: una red tanh ajusta mucho mejor una
    forzante suave que un escalon discontinuo, sin cambiar el caracter cualitativo periodico-pulsado
    de I(t)."""
    phase = torch.remainder(t, I_period) / I_period
    rise = torch.sigmoid(sharpness * phase)
    fall = torch.sigmoid(sharpness * (I_duty - phase))
    return I_amp * rise * fall / norm_factor


T0_state = {'T_u': T0, 'T_uir': T0, 'T_cir': T0, 'T_lir': T0, 'T_l': T0, 'T_cu': T0, 'T_cl': T0}

t_plot_np = np.linspace(0, t_max, 300)
plt.figure(figsize=(8, 2))
plt.plot(t_plot_np, current_waveform(torch.tensor(t_plot_np)).numpy())
plt.xlabel('t [s]'); plt.ylabel('I(t) [A]'); plt.title('Forma de onda de corriente pulsada suavizada (I1=I2)')
plt.show()

## 2. Red PINN: $t\mapsto$ 7 temperaturas de estado (Tabla 1)

In [ ]:
STATE_NAMES = ['T_u', 'T_uir', 'T_cir', 'T_lir', 'T_l', 'T_cu', 'T_cl']

class ThermoelectricPINN(nn.Module):
    def __init__(self, n_hidden=4, n_neurons=48):
        super().__init__()
        layers = [nn.Linear(1, n_neurons), nn.Tanh()]
        for _ in range(n_hidden - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.Tanh()]
        layers += [nn.Linear(n_neurons, 7)]
        self.net = nn.Sequential(*layers)

    def forward(self, t):
        out = self.net(t / t_max)
        # las 7 temperaturas parten todas cerca de T0 (Eq. 14); escala pequena en torno a T0
        # para que la red resuelva bien variaciones de solo unos pocos grados.
        return {name: T0 + out[:, i:i + 1] * 15.0 for i, name in enumerate(STATE_NAMES)}


model = ThermoelectricPINN().to(device)


def d_dt(f, t):
    return torch.autograd.grad(f, t, grad_outputs=torch.ones_like(f),
                                create_graph=True, retain_graph=True)[0]

## 3. Los 7 residuos $\Phi_{phys}$ (Eq. 7-13) y la condicion inicial (Eq. 14)

In [ ]:
N_col = 1500
t_col = torch.linspace(1e-3, t_max, N_col, device=device).view(-1, 1).requires_grad_(True)
t_ic = torch.zeros(1, 1, device=device, requires_grad=True)


def compute_residuals(t):
    T = model(t)
    I1 = current_waveform(t.squeeze(-1)).view(-1, 1)
    I2 = I1  # misma forma de onda para ambas etapas, por simplicidad

    Qu = k1 * (T['T_u'] - T['T_uir']) + T['T_u'] * alpha1 * I1 - 0.5 * R1 * I1**2          # Eq. (1)
    Quir = k1 * (T['T_u'] - T['T_uir']) + T['T_uir'] * alpha1 * I1 + 0.5 * R1 * I1**2       # Eq. (2)
    Qlir = k2 * (T['T_lir'] - T['T_l']) + T['T_lir'] * alpha2 * I2 - 0.5 * R2 * I2**2       # Eq. (3)
    Ql = k2 * (T['T_lir'] - T['T_l']) + T['T_l'] * alpha2 * I2 + 0.5 * R2 * I2**2           # Eq. (4)
    Qloss = W_u * (T0 - T['T_cu'])                                                          # Eq. (5)
    Qout = W_l * (T['T_cl'] - T0)                                                           # Eq. (6)

    dTcu_dt = d_dt(T['T_cu'], t)
    dTcir_dt = d_dt(T['T_cir'], t)
    dTcl_dt = d_dt(T['T_cl'], t)

    R7 = Qloss - Qu - c_u * dTcu_dt                    # Eq. (7)
    R8 = Quir - Qlir - c_ir * dTcir_dt                  # Eq. (8)
    R9 = Qout - Ql + c_l * dTcl_dt                      # Eq. (9)
    R10 = T['T_cu'] - T['T_u'] - r_uu * Qu              # Eq. (10)
    R11 = T['T_uir'] - T['T_cir'] - r_ul * Quir         # Eq. (11)
    R12 = T['T_cir'] - T['T_lir'] - r_lu * Qlir         # Eq. (12)
    R13 = T['T_l'] - T['T_cl'] - r_ll * Ql              # Eq. (13)

    return [R7, R8, R9, R10, R11, R12, R13]


def compute_loss(lam_phys=1.0, lam_ic=20.0):
    residuals = compute_residuals(t_col)
    loss_phys = sum(torch.mean(R**2) for R in residuals)

    T_ic = model(t_ic)
    loss_ic = sum((T_ic[name] - T0_state[name])**2 for name in STATE_NAMES).squeeze()

    return lam_phys * loss_phys + lam_ic * loss_ic, loss_phys.item(), loss_ic.item()

## 4. Entrenamiento

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history = []
for epoch in range(4000):
    optimizer.zero_grad()
    loss, l_phys, l_ic = compute_loss()
    loss.backward()
    optimizer.step()
    history.append(loss.item())
    if epoch % 500 == 0:
        print(f'epoch {epoch:5d} | loss={loss.item():.4e} | phys={l_phys:.4e} | ic={l_ic:.4e}')

## 5. Resultados: evolucion de las 7 temperaturas (efecto de sobre-enfriamiento con corriente pulsada)

In [ ]:
t_test = torch.linspace(0, t_max, 400, device=device).view(-1, 1)
with torch.no_grad():
    T_pred = model(t_test)

plt.figure(figsize=(9, 5))
for name in STATE_NAMES:
    plt.plot(t_test.cpu().numpy(), T_pred[name].cpu().numpy(), label=name)
plt.axhline(T0, color='gray', linestyle=':', label='T0 (ambiente)')
plt.xlabel('t [s]'); plt.ylabel('Temperatura [C]')
plt.title('Evolucion de las 7 temperaturas de estado bajo corriente pulsada')
plt.legend(fontsize=8, ncol=2)
plt.show()

plt.figure(figsize=(6, 4))
plt.semilogy(history)
plt.xlabel('Epoca'); plt.ylabel('Loss (escala log)')
plt.title('Convergencia del funcional $\\Phi_{phys}$')
plt.show()

T_cu_min = T_pred['T_cu'].min().item()
print(f'Temperatura minima alcanzada en el deposito superior (enfriado): {T_cu_min:.2f} C')

**Nota honesta sobre el efecto de sobre-enfriamiento:** en esta configuracion de entrenamiento, la red converge a una solucion con muy poca variacion de temperatura respecto al ambiente (perdida fisica estancada en torno a un valor residual, en vez de decaer varios ordenes de magnitud como en otros cuadernos de esta coleccion). El sistema de 13 ecuaciones (7 residuos: 3 EDOs + 4 relaciones algebraicas, Eq. 7-13) esta correctamente transcrito, pero lograr el efecto de sobre-enfriamiento pronunciado que describe el paper requeriria (como el propio paper senala) ajustar cuidadosamente los pesos individuales $w_i$ de cada residuo y/o usar la forma de onda de corriente realmente optimizada de los trabajos previos [13,14] del paper (aqui se uso una aproximacion representativa). El **mecanismo central -- una unica red con 7 salidas de temperatura, el funcional fisico multi-residuo $\Phi_{phys}$, y la condicion inicial como penalizacion -- esta fielmente implementado**; la demostracion cuantitativa del sobre-enfriamiento por varios grados que reporta el paper no se reprodujo con los parametros representativos usados aqui.